In [1]:
import re
import json
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import pandas_udf

from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

### Create spark session

In [2]:
spark = SparkSession.builder. \
    appName("nyc-jobs-data-exploration"). \
    getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [3]:

import json
import unittest
from unittest.mock import patch
from math import floor

import numpy as np

from pyspark.sql.types import StructType, StringType
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

### Read data

In [4]:
"""
The schema for the input DataFrame is specified in a JSON file.
"""

with open("/dataset/schema/nyc-jobs-schema.json") as schema_file:
    schema = schema_file.read()

nyc_jobs_schema = StructType.fromJson(json.loads(schema))
nyc_jobs_schema

StructType(List(StructField(job_id,IntegerType,false),StructField(agency,StringType,false),StructField(posting_type,StringType,false),StructField(num_of_positions,IntegerType,false),StructField(business_title,StringType,false),StructField(civil_service_title,StringType,false),StructField(title_code_no,StringType,false),StructField(level,StringType,false),StructField(job_category,StringType,true),StructField(ft_pt_indicator,StringType,true),StructField(salary_range_from,DoubleType,false),StructField(salary_range_to,DoubleType,false),StructField(salary_frequency,StringType,false),StructField(work_location,StringType,false),StructField(division_work_unit,StringType,false),StructField(job_description,StringType,false),StructField(min_qual_requirements,StringType,true),StructField(preferred_skills,StringType,true),StructField(additiona_information,StringType,true),StructField(to_apply,StringType,true),StructField(hours_shift,StringType,true),StructField(work_location_1,StringType,true),Stru

In [5]:
file_path = "/dataset/nyc-jobs.csv"
nyc_jobs_df = spark.read.schema(nyc_jobs_schema).\
        option("quote", "\"").\
        option("escape", "\"").\
        option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss.SSS").\
        csv(file_path, header=True)

nyc_jobs_df.printSchema()


root
 |-- job_id: integer (nullable = true)
 |-- agency: string (nullable = true)
 |-- posting_type: string (nullable = true)
 |-- num_of_positions: integer (nullable = true)
 |-- business_title: string (nullable = true)
 |-- civil_service_title: string (nullable = true)
 |-- title_code_no: string (nullable = true)
 |-- level: string (nullable = true)
 |-- job_category: string (nullable = true)
 |-- ft_pt_indicator: string (nullable = true)
 |-- salary_range_from: double (nullable = true)
 |-- salary_range_to: double (nullable = true)
 |-- salary_frequency: string (nullable = true)
 |-- work_location: string (nullable = true)
 |-- division_work_unit: string (nullable = true)
 |-- job_description: string (nullable = true)
 |-- min_qual_requirements: string (nullable = true)
 |-- preferred_skills: string (nullable = true)
 |-- additiona_information: string (nullable = true)
 |-- to_apply: string (nullable = true)
 |-- hours_shift: string (nullable = true)
 |-- work_location_1: string (nu

### Numerical Data Exploration

In [6]:
def calculate_statistics(df, column):
    """
    Calculate min, max, avg, median, and mode for a given column.
    Handles empty DataFrame and missing column safely.
    """

    # Schema for output DataFrame
    result_schema = StructType([
        StructField("min", DoubleType(), True),
        StructField("max", DoubleType(), True),
        StructField("average", DoubleType(), True),
        StructField("median", DoubleType(), True),
        StructField("mode", DoubleType(), True)
    ])

    # Check column exists
    if column not in df.columns:
        return df.sparkSession.createDataFrame(
            [(None, None, None, None, None)], result_schema
        )

    # Remove nulls
    df_clean = df.select(column).filter(F.col(column).isNotNull())

    # Check empty DataFrame
    if df_clean.rdd.isEmpty():
        return df.sparkSession.createDataFrame(
            [(None, None, None, None, None)], result_schema
        )

    # Basic aggregations
    stats_df = df_clean.agg(
        F.min(column).alias("min"),
        F.max(column).alias("max"),
        F.avg(column).alias("average"),
        F.percentile_approx(column, 0.5).alias("median")
    )

    # Mode calculation
    mode_row = (df_clean.groupBy(column)
                        .count()
                        .orderBy(F.desc("count"))
                        .first())

    mode_value = mode_row[0] if mode_row else None

    # Add mode column
    result_df = stats_df.withColumn("mode", F.lit(mode_value))

    return result_df

In [7]:
# Statistics for the numerical column: num_of_positions 
stats_df = calculate_statistics(nyc_jobs_df, "num_of_positions")
stats_df.show()

AttributeError: module 'pyspark.sql.functions' has no attribute 'percentile_approx'

In [ ]:
# Statistics for the numerical column: salary_range_from 
stats_df = calculate_statistics(nyc_jobs_df, "salary_range_from")
stats_df.show()

In [ ]:
# Statistics for the numerical column: salary_range_to 
stats_df = calculate_statistics(nyc_jobs_df, "salary_range_to")
stats_df.show()

### String Data Exploration

In [ ]:
def string_column_statistics(df, column):
    """
    Calculate statistics for a string column:
    - total_count
    - distinct_count
    - null_count
    - empty_count
    - mode (most frequent value)
    - mode_count

    Args:
        df (DataFrame): Input PySpark DataFrame
        column (str): String column name

    Returns:
        DataFrame: Single-row DataFrame with statistics
    """

    # Output schema
    schema = StructType([
        StructField("column_name", StringType(), True),
        StructField("total_count", LongType(), True),
        StructField("distinct_count", LongType(), True),
        StructField("null_count", LongType(), True),
        StructField("empty_count", LongType(), True),
        StructField("mode", StringType(), True),
        StructField("mode_count", LongType(), True)
    ])

    # Column exists check
    if column not in df.columns:
        return df.sparkSession.createDataFrame(
            [(column, None, None, None, None, None, None)], schema
        )

    # Base aggregations
    agg_df = df.agg(
        F.count(column).alias("total_count"),
        F.countDistinct(column).alias("distinct_count"),
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias("null_count"),
        F.sum(F.when(F.col(column) == "", 1).otherwise(0)).alias("empty_count")
    )

    # Mode calculation
    mode_row = (df.filter(F.col(column).isNotNull() & (F.col(column) != ""))
                  .groupBy(column)
                  .count()
                  .orderBy(F.desc("count"))
                  .first())

    mode_value = mode_row[column] if mode_row else None
    mode_count = mode_row["count"] if mode_row else None

    result_df = (agg_df
        .withColumn("column_name", F.lit(column))
        .withColumn("mode", F.lit(mode_value))
        .withColumn("mode_count", F.lit(mode_count))
        .select("column_name","total_count","distinct_count",
                "null_count","empty_count","mode","mode_count")
    )

    return result_df

In [ ]:
"""
Combine all string column statistics into one DataFrame
"""
combined_statistics_df = None
for column, dtype in nyc_jobs_df.dtypes:
    if dtype == "string":
        stats_df = string_column_statistics(nyc_jobs_df, column)
        if combined_statistics_df is None:
            combined_statistics_df = stats_df
        else:
            combined_statistics_df = combined_statistics_df.unionByName(stats_df)

combined_statistics_df.show()
     

In [ ]:
"""
Whats the number of jobs posting per category (Top 10)?
"""

category_distribution_df = (
    nyc_jobs_df
        .groupBy("job_category")
        .agg(F.countDistinct("job_id").alias("num_job_postings"))
        .orderBy(F.desc("num_job_postings"))
        .limit(10)
)
category_distribution_df.show(truncate=False)

In [ ]:
"""
Whats the salary distribution per job category?
"""
distribution_df = (
    nyc_jobs_df
        .groupBy("job_category")
        .agg(
            F.min("salary_range_from").alias("min_salary_from"),
            F.max("salary_range_from").alias("max_salary_from"),
            F.avg("salary_range_from").alias("avg_salary_from"),
            F.min("salary_range_to").alias("min_salary_to"),
            F.max("salary_range_to").alias("max_salary_to"),
            F.avg("salary_range_to").alias("avg_salary_to")
        )
        .fillna({
            "min_salary_from": 0,
            "max_salary_from": 0,
            "avg_salary_from": 0,
            "min_salary_to": 0,
            "max_salary_to": 0,
            "avg_salary_to": 0
        })
        .select(
            "job_category",
            "min_salary_from",
            "min_salary_to",
            "max_salary_from",
            "max_salary_to",
            "avg_salary_from",
            "avg_salary_to"
        )
)

distribution_df.show(truncate=True)

In [ ]:
"""
Is there any correlation between the higher degree and the salary?
"""
DEGREE_KEYWORDS = ["baccalaureate", "graduate", "master", "ph.d.", "high school", "university", "undergraduate"]

def get_highest_degree(degrees_string: str) -> str:
    """
    Return the highest degree based on predefined precedence.
    """

    if not degrees_string:
        return None

    degree_precedence = [
        "Ph.D.", "master", "graduate", "baccalaureate",
         "undergraduate", "university", "high school"
    ]

    # Create ranking map for fast lookup
    degree_rank = {degree: rank for rank, degree in enumerate(degree_precedence)}

    #degrees = [d.strip() for d in degrees_string.lower()]
    valid_degrees = [degree for degree in degree_rank if degree.lower() in degrees_string.lower()]

    # valid_degrees = [d for d in degrees if d.lower() in degree_rank]

    if not valid_degrees:
        return None

    return min(valid_degrees, key=lambda d: degree_rank[d])
    
@pandas_udf(StringType())
def extract_qualification_degree_udf(text_series: pd.Series) -> pd.Series:
    def extract_degree(text):
        if pd.isna(text):
            return "N/A"

        tokens = re.findall(r"high school|\w+(?:\.\w+)*\.?", text.lower())
        matched = [kw for kw in DEGREE_KEYWORDS if any(kw in token for token in tokens)]

        if not matched:
            return "N/A"

        return get_highest_degree(",".join(matched))

    return text_series.apply(extract_degree)

df_with_degree = (
    nyc_jobs_df
        .filter(F.col("min_qual_requirements").isNotNull())
        .withColumn("degree", extract_qualification_degree_udf(F.col("min_qual_requirements")))
)
# # Degree precedence mapping
degree_precedence = {
    "Ph.D.": 7,
    "master": 6,
    "graduate": 5,
    "baccalaureate": 4,
    "undergraduate": 3,
    "university": 3,
    "high school": 1,
    "N/A": 0
}

# Convert dict to Spark map expression
mapping_expr = F.create_map([F.lit(x) for x in sum(degree_precedence.items(), ())])

df_with_degree_numeric = (
    df_with_degree
        .withColumn("degree_numeric", mapping_expr[F.col("degree")].cast("int"))
        .filter(F.col("degree_numeric").isNotNull())
)
# df_with_degree_numeric.select("degree", "degree_numeric").where("degree_numeric>0").show()
correlation = df_with_degree_numeric.stat.corr("degree_numeric", "salary_range_to")
print(f"Correlation between degree level and salary_range_to: {correlation:.4f}")

print(
    f"{correlation:.4f} indicates a weak negative correlation, "
    "suggesting that higher education levels are slightly associated with higher salaries."
)


In [ ]:
# Whats the job posting having the highest salary per agency?


window_spec = Window.partitionBy("agency").orderBy(F.desc("salary_range_to"))

highest_salary_jobs_df = (
    nyc_jobs_df
        .withColumn("rank", F.dense_rank().over(window_spec))
        .filter(F.col("rank") == 1)
        .select("job_id", "agency", "salary_range_to")
        .distinct()
)

highest_salary_jobs_df.show(truncate=False)

In [ ]:
"""
Whats the job positings average salary per agency for the last 2 years?
"""

max_posting_date = nyc_jobs_df.select(F.max("posting_date")).first()[0]
last_two_years_df = nyc_jobs_df.filter(
    F.col("posting_date") >= F.add_months(F.lit(max_posting_date), -24)
)

avg_salary_per_agency_df = (
    last_two_years_df
        .groupBy("agency")
        .agg(F.avg("salary_range_to").alias("avg_salary_per_agency"))
        .orderBy(F.desc("avg_salary_per_agency"))
)

avg_salary_per_agency_df.show(truncate=False)

In [ ]:
"""
What are the highest paid skills in the US market?
"""
highest_paying_skills_df = (
    nyc_jobs_df
        .groupBy("preferred_skills")
        .agg(F.avg("salary_range_to").alias("average_salary"))
        .orderBy(F.desc("average_salary"))
        .limit(10)
)

highest_paying_skills_df.show()

### Data Processing

In [ ]:
"""
Data Cleaning - Remove duplicate records and replace null values with default placeholders
"""

cleaned_df = (
    nyc_jobs_df
        .dropDuplicates()
        .fillna({
            "job_id": 0,
            "num_of_positions": 0,
            "salary_range_from": 0,
            "salary_range_to": 0
        })
)


In [ ]:
"""
Data Cleaning - Handle nulls & trim strings
"""
def clean_string_columns(df: DataFrame) -> DataFrame:
    string_cols = [c for c, t in df.dtypes if t == "string"]

    for col in string_cols:
        df = df.withColumn(col, F.trim(F.col(col)))

    return df

cleaned_df = clean_string_columns(cleaned_df)


In [ ]:
"""
normalize text columns
"""
def normalize_text_columns(df: DataFrame) -> DataFrame:
    string_cols = [c for c, t in df.dtypes if t == "string"]

    for col in string_cols:
        df = df.withColumn(col, F.lower(F.col(col)))

    return df
normalised_df = normalize_text_columns(cleaned_df)


In [ ]:
"""
Degree Extraction usng Vectorized UDF
"""
df_with_degree = (
    normalised_df
        .filter(F.col("min_qual_requirements").isNotNull())
        .withColumn("degree", extract_qualification_degree_udf(F.col("min_qual_requirements")))
)


In [ ]:
"""
Add average salary
"""
def add_average_salary(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "avg_salary",
        (F.col("salary_range_from") + F.col("salary_range_to")) / 2
    )
df_with_avg_salary = add_average_salary(df_with_degree)

In [ ]:
"""
Salary binning - categorical features
Apply optimal salary binning on salary_range_from
"""

salary_bin_expr = (
    F.when(F.col("salary_range_from") < 30000, "<30K")
     .when(F.col("salary_range_from").between(30000, 50000), "30K–50K")
     .when(F.col("salary_range_from").between(50001, 75000), "50K–75K")
     .when(F.col("salary_range_from").between(75001, 100000), "75K–100K")
     .when(F.col("salary_range_from").between(100001, 150000), "100K–150K")
     .otherwise(">150K")
)

salary_binned_df = df_with_avg_salary.withColumn("salary_band", salary_bin_expr)

salary_binned_df.groupBy(["job_category", "salary_band"]).agg(F.countDistinct("job_id").alias("number_of_jobs")).show()


In [ ]:
# Date-based features 
date_based_features_df = (
    salary_binned_df
        .withColumn("posting_year", F.year("posting_date"))
        .withColumn("posting_month", F.month("posting_date"))
        .withColumn("posting_day", F.dayofmonth("posting_date"))
)


In [ ]:
"""
Data Cleaning - Remove non-essential columns from the dataset
"""

# Columns identified for removal based on profiling results
columns_to_drop = [
    "recruitment_contact",
    "process_date",
    "posting_updated",
    "job_description"
]

final_nyc_df = date_based_features_df.drop(*columns_to_drop)

final_nyc_df.printSchema()

In [ ]:
"""
Persist the transformed dataset to the target storage location in partitioned Parquet format
"""

(
    final_nyc_df
        .write
        .mode("overwrite")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .partitionBy("agency")
        .parquet("/dataset/processed/")
)

### Test Cases

In [ ]:
import unittest
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
import warnings
warnings.filterwarnings("ignore", category=ResourceWarning)


class TestCalculateStatistics(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder \
            .master("local[1]") \
            .appName("UnitTestCalculateStatistics") \
            .getOrCreate()

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_valid_column_statistics(self):
        """Test statistics on a valid numeric column"""

        data = [(1,), (2,), (2,), (3,), (4,)]
        schema = StructType([StructField("value", IntegerType(), True)])
        df = self.spark.createDataFrame(data, schema)

        result_df = calculate_statistics(df, "value")
        result = result_df.collect()[0]

        self.assertEqual(result["min"], 1)
        self.assertEqual(result["max"], 4)
        self.assertAlmostEqual(result["average"], 2.4)
        self.assertEqual(result["median"], 2)
        self.assertEqual(result["mode"], 2)

    def test_column_not_exists(self):
        """Test when column does not exist"""

        data = [(1,), (2,)]
        df = self.spark.createDataFrame(data, ["value"])

        result_df = calculate_statistics(df, "salary")
        result = result_df.collect()[0]

        self.assertIsNone(result["min"])
        self.assertIsNone(result["max"])
        self.assertIsNone(result["average"])
        self.assertIsNone(result["median"])
        self.assertIsNone(result["mode"])

    def test_empty_dataframe(self):
        """Test with empty DataFrame"""

        schema = StructType([StructField("value", DoubleType(), True)])
        df = self.spark.createDataFrame([], schema)

        result_df = calculate_statistics(df, "value")
        result = result_df.collect()[0]

        self.assertIsNone(result["min"])
        self.assertIsNone(result["max"])
        self.assertIsNone(result["average"])
        self.assertIsNone(result["median"])
        self.assertIsNone(result["mode"])

    def test_dataframe_with_nulls_only(self):
        """Test when column contains only null values"""

        data = [(None,), (None,)]
        schema = StructType([StructField("value", DoubleType(), True)])
        df = self.spark.createDataFrame(data, schema)

        result_df = calculate_statistics(df, "value")
        result = result_df.collect()[0]

        self.assertIsNone(result["min"])
        self.assertIsNone(result["max"])
        self.assertIsNone(result["average"])
        self.assertIsNone(result["median"])
        self.assertIsNone(result["mode"])

    def test_single_value_dataframe(self):
        """Test when DataFrame has only one value"""

        data = [(10,)]
        schema = StructType([StructField("value", IntegerType(), True)])
        df = self.spark.createDataFrame(data, schema)

        result_df = calculate_statistics(df, "value")
        result = result_df.collect()[0]

        self.assertEqual(result["min"], 10)
        self.assertEqual(result["max"], 10)
        self.assertEqual(result["average"], 10)
        self.assertEqual(result["median"], 10)
        self.assertEqual(result["mode"], 10)
        

suite = unittest.TestLoader().loadTestsFromTestCase(TestCalculateStatistics)
unittest.TextTestRunner().run(suite)

In [ ]:
class TestStringColumnStatistics(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = SparkSession.builder.master("local[1]").getOrCreate()
        import warnings
        warnings.filterwarnings("ignore", category=ResourceWarning)

    def test_all_nulls(self):
        """FIXED: Explicitly define the schema so Spark doesn't fail to infer type"""
        # Define a schema so Spark knows 'only_nulls' is a String column
        schema = StructType([StructField("only_nulls", StringType(), True)])
        df = self.spark.createDataFrame([(None,), (None,)], schema)
        
        result = string_column_statistics(df, "only_nulls").collect()[0]

        self.assertEqual(result["null_count"], 2)
        self.assertEqual(result["total_count"], 0) # total_count (F.count) ignores nulls
        self.assertIsNone(result["mode"])

    def test_happy_path(self):
        """FIXED: Adjusting expectation for total_count"""
        data = [("A"), ("A"), ("B"), (""), (None)]
        # Use StringType() during creation to be safe
        df = self.spark.createDataFrame(data, StringType()).toDF("test_col")

        result = string_column_statistics(df, "test_col").collect()[0]

        # Total count = 4 (A, A, B, and "") because "" is NOT null.
        self.assertEqual(result["total_count"], 4) 
        
        # Distinct count = 3 ("A", "B", "")
        self.assertEqual(result["distinct_count"], 3)
        
        self.assertEqual(result["null_count"], 1)
        self.assertEqual(result["empty_count"], 1)
        self.assertEqual(result["mode"], "A")
        self.assertEqual(result["mode_count"], 2)

    def test_empty_column_logic(self):
        """Verify behavior with ONLY empty strings"""
        data = [(""), (""), ("")]
        df = self.spark.createDataFrame(data, StringType()).toDF("empties")
        
        result = string_column_statistics(df, "empties").collect()[0]
        
        self.assertEqual(result["total_count"], 3) # "" is counted
        self.assertEqual(result["empty_count"], 3)
        self.assertIsNone(result["mode"]) # Because your function filters out "" from mode

suite = unittest.TestLoader().loadTestsFromTestCase(TestStringColumnStatistics)
unittest.TextTestRunner().run(suite)

In [ ]:
class TestGetHighestDegree(unittest.TestCase):

    def test_single_degree(self):
        self.assertEqual(get_highest_degree("master"), "master")

    def test_multiple_degrees_precedence(self):
        degrees = "undergraduate master Ph.D."
        self.assertEqual(get_highest_degree(degrees), "Ph.D.")

    def test_order_independent(self):
        degrees = "high school graduate baccalaureate"
        self.assertEqual(get_highest_degree(degrees), "graduate")

    def test_invalid_degrees(self):
        degrees = "diploma certificate"
        self.assertIsNone(get_highest_degree(degrees))

    def test_empty_string(self):
        self.assertIsNone(get_highest_degree(""))

    def test_none_input(self):
        self.assertIsNone(get_highest_degree(None))

    def test_extra_spaces(self):
        degrees = "   master    undergraduate   "
        self.assertEqual(get_highest_degree(degrees), "master")

    def test_case_sensitive_match(self):
        degrees = "Ph.D. master"
        self.assertEqual(get_highest_degree(degrees), "Ph.D.")


suite = unittest.TestLoader().loadTestsFromTestCase(TestGetHighestDegree)
unittest.TextTestRunner().run(suite)

In [ ]:

class TestCleanStringColumns(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        """Start Spark once for the test session."""
        warnings.filterwarnings("ignore", category=ResourceWarning)
        cls.spark = SparkSession.builder \
            .master("local[1]") \
            .appName("test") \
            .getOrCreate()
        cls.spark.sparkContext.setLogLevel("ERROR")

    @classmethod
    def tearDownClass(cls):
        # Give Spark a moment to finish accumulator updates
        import time
        time.sleep(1) 
        cls.spark.stop()

    def test_trimming_logic(self):
        """Verify that leading/trailing spaces are removed from strings."""
        data = [("  Software Engineer", "Ph.D.  ", "  active  ")]
        df = self.spark.createDataFrame(data, ["title", "degree", "status"])

        # WHEN: Applying the cleaning function
        df_cleaned = clean_string_columns(df)
        result = df_cleaned.collect()[0]

        # THEN: All values should be trimmed
        self.assertEqual(result["title"], "Software Engineer")
        self.assertEqual(result["degree"], "Ph.D.")
        self.assertEqual(result["status"], "active")

    def test_mixed_types(self):
        """Ensure that non-string columns (like Integer) are not affected."""
        schema = StructType([
            StructField("name", StringType(), True),
            StructField("age", IntegerType(), True)
        ])
        data = [("  Alice  ", 30)]
        df = self.spark.createDataFrame(data, schema)

        # WHEN: Applying the cleaning function
        df_cleaned = clean_string_columns(df)
        result = df_cleaned.collect()[0]

        # THEN: String is trimmed, Integer remains the same
        self.assertEqual(result["name"], "Alice")
        self.assertEqual(result["age"], 30)

    def test_empty_and_null_strings(self):
        """FIXED: Define the schema to avoid Type Inference errors."""
        from pyspark.sql.types import StructType, StructField, StringType
        
        # Explicitly tell Spark these are String columns
        schema = StructType([
            StructField("whitespace_only", StringType(), True),
            StructField("null_col", StringType(), True)
        ])
        
        data = [("   ", None)]
        df = self.spark.createDataFrame(data, schema)
    
        # WHEN: Applying the cleaning function
        df_cleaned = clean_string_columns(df)
        result = df_cleaned.collect()[0]
    
        # THEN: All assertions should now pass
        self.assertEqual(result["whitespace_only"], "")
        self.assertIsNone(result["null_col"])


suite = unittest.TestLoader().loadTestsFromTestCase(TestCleanStringColumns)
unittest.TextTestRunner().run(suite)

In [ ]:
class TestNormalizeTextColumns(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        """Start Spark once and silence ResourceWarnings."""
        warnings.filterwarnings("ignore", category=ResourceWarning)
        cls.spark = SparkSession.builder \
            .master("local[1]") \
            .appName("TestNormalization") \
            .getOrCreate()
        # Ensure console stays clean from Spark INFO logs
        cls.spark.sparkContext.setLogLevel("ERROR")

    @classmethod
    def tearDownClass(cls):
        """Stop Spark after all tests finish."""
        cls.spark.stop()

    def test_lowercase_conversion(self):
        """Verify that strings are correctly lowercased."""
        data = [("Software Engineer", "PH.D."), ("DATA SCIENTIST", "Master")]
        df = self.spark.createDataFrame(data, ["title", "degree"])

        # WHEN: Applying normalization
        df_norm = normalize_text_columns(df)
        results = df_norm.collect()

        # THEN: Check values
        self.assertEqual(results[0]["title"], "software engineer")
        self.assertEqual(results[0]["degree"], "ph.d.")
        self.assertEqual(results[1]["title"], "data scientist")
        self.assertEqual(results[1]["degree"], "master")

    def test_skips_non_string_columns(self):
        """Ensure Integer columns are not touched."""
        schema = StructType([
            StructField("name", StringType(), True),
            StructField("salary", IntegerType(), True)
        ])
        data = [("ALICE", 150000)]
        df = self.spark.createDataFrame(data, schema)

        # WHEN: Applying normalization
        df_norm = normalize_text_columns(df)
        result = df_norm.collect()[0]

        # THEN: Name is lowercased, Salary is still an Integer
        self.assertEqual(result["name"], "alice")
        self.assertEqual(result["salary"], 150000)

    def test_null_handling(self):
        """Verify that Null values do not cause crashes."""
        schema = StructType([StructField("col_a", StringType(), True)])
        df = self.spark.createDataFrame([(None,)], schema)

        # WHEN: Applying normalization
        df_norm = normalize_text_columns(df)
        result = df_norm.collect()[0]

        # THEN: Null remains Null
        self.assertIsNone(result["col_a"])

suite = unittest.TestLoader().loadTestsFromTestCase(TestNormalizeTextColumns)
unittest.TextTestRunner().run(suite)


In [ ]:
class TestSalaryCalculations(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        """Start Spark once and suppress resource warnings."""
        warnings.filterwarnings("ignore", category=ResourceWarning)
        cls.spark = SparkSession.builder \
            .master("local[1]") \
            .getOrCreate()
        cls.spark.sparkContext.setLogLevel("ERROR")

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_salary_average_math(self):
        """Verify the calculation: (from + to) / 2."""
        data = [(100000, 150000), (80000, 100000)]
        df = self.spark.createDataFrame(data, ["salary_range_from", "salary_range_to"])

        # WHEN: Calculating average
        df_result = add_average_salary(df)
        results = df_result.collect()

        # THEN: 125000.0 and 90000.0
        self.assertEqual(results[0]["avg_salary"], 125000.0)
        self.assertEqual(results[1]["avg_salary"], 90000.0)

    def test_null_salary_handling(self):
        """Verify that if one side is Null, the result is Null (standard Spark behavior)."""
        schema = StructType([
            StructField("salary_range_from", IntegerType(), True),
            StructField("salary_range_to", IntegerType(), True)
        ])
        # Case where one or both columns are Null
        data = [(100000, None), (None, None)]
        df = self.spark.createDataFrame(data, schema)

        # WHEN: Calculating average
        df_result = add_average_salary(df)
        results = df_result.collect()

        # THEN: Math with Null should result in Null
        self.assertIsNone(results[0]["avg_salary"])
        self.assertIsNone(results[1]["avg_salary"])

    def test_zero_values(self):
        """Verify the logic holds with zero values."""
        data = [(0, 100)]
        df = self.spark.createDataFrame(data, ["salary_range_from", "salary_range_to"])
        
        result = add_average_salary(df).collect()[0]
        self.assertEqual(result["avg_salary"], 50.0)
suite = unittest.TestLoader().loadTestsFromTestCase(TestSalaryCalculations)
unittest.TextTestRunner().run(suite)

In [ ]:
class TestQualificationDegreeUDF(unittest.TestCase):
  
    @classmethod
    def setUpClass(cls):
        """Start Spark once for the test session."""
        warnings.filterwarnings("ignore", category=ResourceWarning)
        cls.spark = SparkSession.builder \
            .master("local[1]") \
            .appName("test") \
            .getOrCreate()
        cls.spark.sparkContext.setLogLevel("ERROR")

    @classmethod
    def tearDownClass(cls):
        # Give Spark a moment to finish accumulator updates
        import time
        time.sleep(1) 
        cls.spark.stop()
    
    def test_valid_degrees(self):
        data = [
            ("I have a master degree",),
            ("Completed Ph.D. in AI",),
            ("high school education",),
        ]
        schema = "min_qual_requirements STRING"
        df = self.spark.createDataFrame(data, schema=schema)
        result = df.withColumn(
            "degree", extract_qualification_degree_udf(df["min_qual_requirements"])
        ).collect()
        self.assertEqual(result[0]["degree"], "master")
        self.assertEqual(result[1]["degree"], "Ph.D.")
        self.assertEqual(result[2]["degree"], "high school")
    
    def test_multiple_degrees(self):
        data = [("undergraduate and master degree",)]
        schema = "min_qual_requirements STRING"
        df = self.spark.createDataFrame(data, schema=schema)
        result = df.withColumn(
            "degree", extract_qualification_degree_udf(df["min_qual_requirements"])
        ).collect()[0]["degree"]
        self.assertEqual(result, "master")
    
    def test_no_degree_found(self):
        data = [("I love programming",)]
        schema = "min_qual_requirements STRING"
        df = self.spark.createDataFrame(data, schema=schema)
        result = df.withColumn(
            "degree", extract_qualification_degree_udf(df["min_qual_requirements"])
        ).collect()[0]["degree"]
        self.assertEqual(result, "N/A")
    
    def test_null_value(self):
        data = [(None,)]
        schema = "min_qual_requirements STRING"
        df = self.spark.createDataFrame(data, schema=schema)
        result = df.withColumn(
            "degree", extract_qualification_degree_udf(df["min_qual_requirements"])
        ).collect()[0]["degree"]
        self.assertEqual(result, "N/A")
    
    def test_mixed_input(self):
        data = [
            ("Ph.D. and master degree",),
            ("graduate level",),
            ("",),
            (None,)
        ]
        schema = "min_qual_requirements STRING"
        df = self.spark.createDataFrame(data, schema=schema)
        results = df.withColumn(
            "degree", extract_qualification_degree_udf(df["min_qual_requirements"])
        ).collect()
        self.assertEqual(results[0]["degree"], "Ph.D.")
        self.assertEqual(results[1]["degree"], "graduate")
        self.assertEqual(results[2]["degree"], "N/A")
        self.assertEqual(results[3]["degree"], "N/A")

# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestQualificationDegreeUDF)
unittest.TextTestRunner().run(suite)